# Annotation
This notebook is used to annotate LLM's reasoning traces with our taxonomy and validate the performance of the LLM-annotator.

In [ ]:
import os
import json
import glob
import re
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
from tqdm import tqdm

from openai import OpenAI

import pandas as pd
from pathlib import Path

from src.datasets import get_or_create_dataset
from src.equality import SemanticDoubleEqualityChecker
from src.model_configurations import gpt_4_1, gpt_4_1_mini_det_config, gpt_5_mini_config
from src.prompt_util import prompt_openai

load_dotenv()

In [ ]:
eedi_dataset = get_or_create_dataset("eedi_data", n_limit=500)
print(f"We have {len(eedi_dataset)} EEDI questions")

datasets_by_datafolder = {
    "eedi_data": eedi_dataset
}

equality_model_config = gpt_4_1_mini_det_config
expert_equality_model_config = gpt_5_mini_config
equality_client = OpenAI(base_url=equality_model_config["base_url"], api_key=os.environ.get(equality_model_config["api_key_var"], None))
semantic_equality_checker = SemanticDoubleEqualityChecker(equality_client, equality_model_config, expert_equality_model_config)

In [ ]:
TAXONOMY = """
<INTER>
Definition: Reasoning about the task instructions or requirements — what the question asks for and what counts as valid answers.  
Rules:
- Only mark when the expert revisits the task description and subsequently tries to gain clarity about the task itself. 
- Do NOT mark execution steps, calls to produce output, or listing candidates (e.g., "I'll produce:", "Let's do:", "Distractor1: 0.4<INST>"). 
Examples:
- "We are given the question: ..."
- "However<RECON>, the task is to generate three incorrect distractors, not the correct answer<INTER>"

<CORR>
Definition: Correct computation or reasoning toward the correct solution for the question.
Rules:
- Mark whenever correct reasoning or the correct answer is referenced.
- If correct reasoning and errors are discussed together, mark both.
Examples:
- "2 ÷ 1/5 = 10<CORR>"
- "Multiplying both sides by 4 gives 20 = k<CORR>, but a student might only multiply the numerator<ERR_DESC>"

<ERR_DESC>
Definition: High-level verbal description of a common mistake or misconception.
Rules:
- Mark every description of an error.
Examples:
- "A common mistake is forgetting to flip the fraction<ERR_DESC>"
- "46 <INST> (forgetting to add 2)<ERR_DESC>"
- "(x,y)=(-2,15)<INST> [from sign error<ERR_DESC>]"
- "Mis-handling the negative<ERR_DESC>: -10 + 8 <ERR_SIM> = 2<INST>"

<ERR_SIM>
Definition: Explicitly simulating incorrect reasoning.
Rules:
- Mark when the expert simulates an incorrect calculation.
- Single incorrect equations can be marked if they represent erroneous reasoning.
- Mark the final incorrect outcome with <INST>.
- ERR_DESC = a high level error description; ERR_SIM = a specific execution of an error
Examples:
- "5 - 2 = 3, then add 1 = <ERR_SIM> 4<INST>"
- "9 + 3 = 12, write down 2, forget to carry the 1… final result <ERR_SIM> 82<INST>"
- "Convert the fraction incorrectly <ERR_DESC>: compute: 1 2/3 <ERR_SIM><INST>"

<INST>
Definition: Any incorrect outcome (number, symbol, expression).
Rules:
- Mark every candidate, even if later rejected.
- Mark candidate values even when they appear inside task interpretation or reconsideration spans, as long as they name concrete answer options
- Each value in an enumeration of candidates is marked separately; enumeration markers like 1., 2., 3. are NOT tagged.
Examples:
- "0.4<INST>, 0.1<INST>, 2.5<INST>"
- "Possible answers could be Alice <INST>, Bob <INST>, etc"
- "980<INST> might work"
- "(x,y)=(-2,15)<INST> [from sign error<ERR_DESC>]"

<PLAUS>
Definition: Judgment of how likely a student would choose an error or candidate.
Rules:
- Mark plausibility comparisons or checks for incorrectness.
- If also about final set, mark both PLAUS and CURATE.
Examples:
- "0.4<INST> is more plausible than 0.1<INST><PLAUS>"
- "The student forgets to add?<ERR_DESC> Plausibly<PLAUS>"
- "0.4<INST> is not a good distractor<PLAUS>"
- "But is a student going to make that mistake?<PLAUS>"

<CURATE>
Definition: Evaluation or selection of the final set of distractors (coverage, diversity, redundancy).
Rules:
- Only mark when reasoning explicitly concerns the final set.
- Otherwise, mark PLAUS.
Examples:
- "Keep 0.4<INST> and 2.5<INST>, drop 0.1<INST> to cover error types<CURATE>"
- "0.4<INST> seems plausible<PLAUS>, keep that<CURATE>"

<RECON>
Definition: Reconsideration of a previous interpretation, candidate, plausibility judgment, or curation decision.
Rules:
- Place <RECON> immediately after the cue word indicating reconsideration.
- Marks the act of reconsidering, not the outcome.
- Common cues: "actually", "alternatively", "instead", "however", "but wait", "on second thought", "reconsider"
Examples:
- "Actually<RECON>, ..."
- "Alternatively<RECON>, 980<INST> could work<PLAUS>"
- "On second thought<RECON>, that distractor is not likely<PLAUS>"
"""


def extract_examples_from_trace(client: OpenAI, model_config: dict, rt: str, max_examples: int = 3) -> dict:
    """Return taxonomy placeholder and a raw examples string for the trace.
    """
    system_prompt = f"""
You are a helper that extracts up to {max_examples} short example spans for each taxonomy label from a single reasoning trace.
For each tag, return a short examples block (plain text) with one line per tag in the following form:
<TAG>: example1; example2
If no examples exist for a tag, use: <TAG>: (none)
Return only the plain text block (no JSON, no commentary).


TAXONOMY:
{TAXONOMY}
"""
    user_prompt = f"""TRACE START
{rt}
TRACE END

Return only the examples block as described."""
    try:
        examples_raw = prompt_openai(client, system_prompt, user_prompt, model_config)
    except Exception as e:
        print(f"Example extraction failed: {e}")
        return ""

    return examples_raw

def chunk_text(text: str, max_len: int = 2000, min_len: int = 500) -> list:
    """Split text into chunks near max_len using \\n\\n as soft boundaries.
    No overlap. Returns list of tuples: (start_index, chunk_text).
    """
    if len(text) <= max_len:
        return [(0, text)]

    chunks = []
    blocks = text.split("\n\n")

    current = ""
    start = 0
    cursor = 0  # position in original text

    for i, block in enumerate(blocks):
        sep = "\n\n" if i < len(blocks) - 1 else ""
        piece = block + sep

        # Hard split if a single block is too large
        if len(piece) > max_len:
            if current:
                chunks.append((start, current))
                start += len(current)
                current = ""

            offset = 0
            while offset < len(piece):
                chunks.append((cursor + offset, piece[offset:offset + max_len]))
                offset += max_len

            cursor += len(piece)
            start = cursor
            continue

        # If adding this would exceed max_len and we're "big enough", flush
        if current and len(current) + len(piece) > max_len and len(current) >= min_len:
            chunks.append((start, current))
            start += len(current)
            current = piece
        else:
            current += piece

        cursor += len(piece)

    if current:
        chunks.append((start, current))

    return chunks

def annotate_rt(client: OpenAI, model_config: dict, rt: str, fallback_model_config: dict | None = None,
                chunk_max_len: int = 1000, overlap: int = 200) -> dict:
    """Chunk-based annotator.

    Steps:
      1) extract short examples for each label from the full trace
      2) split the trace into chunks (< chunk_max_len)
      3) iteratively prompt the model to annotate each chunk (insert opening tags only)
      4) collect parsed tag positions (offset to original text), merge and reconstruct final annotated trace

    Returns dict with keys: 'merged' (final annotated text), 'per_chunk' (list of chunk annotation texts),
    'examples' (extracted examples), 'merged_items' (merged events)
    """
    # 1) extract examples
    examples_raw = extract_examples_from_trace(client, model_config, rt, max_examples = 3)

    # 2) split into chunks
    chunks = chunk_text(rt, max_len=chunk_max_len)

    # Iteratively annotate chunks
    per_chunk = []
    for start, chunk in chunks:
        system_prompt = f"""
You are annotating a chunk of a thinking-out-loud protocol produced by an expert model with markers of a taxonomy.

Context:
The text is a verbalized reasoning trace of an expert generating incorrect distractor answers for a mathematics multiple-choice question.

The expert's task was:
"You will be given a math question. Please generate 3 incorrect distractor answers for the question to be used as multiple-choice options in a multiple-choice exam."

The protocol contains the expert's internal reasoning, planning, and candidate generation steps.
Your job is to annotate this text by inserting taxonomy tags.

Each marker marks the END of the smallest possible span that instantiates the category.

TAXONOMY
{TAXONOMY}

EXAMPLES:
{examples_raw}
"""
        user_prompt = f"""CHUNK START
{chunk}
CHUNK END

Return only the annotated chunk (no explanations).
"""
        try:
            ann_chunk = prompt_openai(client, system_prompt, user_prompt, model_config).replace("CHUNK START", "").replace("CHUNK END", "")
        except Exception as e:
            print(f"Chunk annotation failed at start={start}: {e}")
            ann_chunk = chunk  # fallback to raw chunk (no tags)

        per_chunk.append(ann_chunk)

    return "\n\n".join(per_chunk)

### Annotate a Subset for Manual Review

In [ ]:
def annotate_and_save(args):
    client, model_config, fallback_model_config, reasoning, out_path = args
    try:
        annotator = model_config["model"]
        answer = annotate_rt(client, model_config, reasoning)
        if len(answer) < 10:
            print(f"Retrying with chat model because reasoning model most likely ran out of space...")
            annotator = fallback_model_config["model"]
            answer = annotate_rt(client, fallback_model_config, reasoning)
        
        with open(out_path, "w+") as f:
            f.write(f"ANNOTATOR: {annotator}\n")
            f.write(answer)
    except Exception as e:
        print(e)
    return out_path


def annotate_for_manual_review(run_name: str, n_samples: int = 2, model_config = gpt_4_1, is_reasoning: bool = True):
    with open(f"eedi_data/joint_results/{run_name}_responses_by_datapointid.json", "r") as f:
        responses = json.load(f)

    results_df = pd.read_csv(f"eedi_data/joint_results/{run_name}_results.csv")

    datapoints = []

    for k,response in responses.items():
        dp = eedi_dataset[int(k)]
        try:
            result = results_df[results_df["Id"] == int(k)].iloc[0]
        except:
            print(f"Could not find {k} in {set(results_df['Id'])}")
        
        trace = response.get("raw_reasoning", "")
        if not is_reasoning:
            trace = response.get("step_by_step")
        datapoints.append((k, dp["Problem"]["Question"], trace, result["proportional_match"], result["number_correct"], result["repetitions"], result["distractors"], dp["Problem"]["Solvable"], result["num_cor_sol_steps_by_datapointid"]))

    df = pd.DataFrame(datapoints, columns=["Id", "problem", "trace", "proportional_match", "number_correct", "repetitions", "distractors", "solvable", "nr_steps_of_solution"])

    unsolvable_df = df[~df["solvable"]]

    high_match_solvable_df = df[(df["proportional_match"] > 0.5) & (df["solvable"])]
    low_match_solvable_df = df[(df["proportional_match"] < 0.5)  & (df["solvable"])]

    high_match_long_problems_solvable_df = df[(df["proportional_match"] > 0.5) & (df["nr_steps_of_solution"] > 4) & (df["solvable"])]
    low_match_long_problems_solvable_df = df[(df["proportional_match"] < 0.5) & (df["nr_steps_of_solution"] > 4) & (df["solvable"])]

    print(len(unsolvable_df), len(high_match_solvable_df), len(low_match_solvable_df), len(high_match_long_problems_solvable_df), len(low_match_long_problems_solvable_df))

    client = OpenAI(base_url=model_config["base_url"], api_key=os.environ.get(model_config["api_key_var"], None))

    os.makedirs(f"manual_inspection/tagging_evaluation/joint/eedi_data/{run_name}", exist_ok=True)

    tasks = []
    for category, df_cat in {
        # "unsolvable": unsolvable_df, 
        "high_match_solvable": high_match_solvable_df, 
        "low_match_solvable": low_match_solvable_df, 
        "high_match_long_problems_solvable": high_match_long_problems_solvable_df, 
        "low_match_long_problems_solvable": low_match_long_problems_solvable_df
    }.items():
        samples = df_cat.sample(n_samples)
        for i, (_, sample) in enumerate(samples.iterrows()):
            out_path = f"manual_inspection/tagging_evaluation/joint/eedi_data/{run_name}/{category}_{i}.txt"
            tasks.append((client, model_config, model_config, sample["trace"], out_path)) # always use chat

    with ThreadPoolExecutor(max_workers=8) as executor:
        list(tqdm(executor.map(annotate_and_save, tasks), total=len(tasks)))

In [ ]:
annotate_for_manual_review("deepseek-naive-deepseek-reasoner", n_samples=2)
annotate_for_manual_review("deepseek-naive-cot-deepseek-chat", n_samples=2, is_reasoning=False)
annotate_for_manual_review("openrouter-naive-z-ai_glm-4.7-reasoner", n_samples=2)
annotate_for_manual_review("openrouter-naive-cot-z-ai_glm-4.7-chat", n_samples=2, is_reasoning=False)

### Agreement with Manual Annotation

In [ ]:
def load_manual_annotations(dirpath: str):
    """Load all .txt files in dirpath and extract tags with optional +/- prefixes.
    Returns dict: filepath -> list of matches {'pos','sign','label','file'}
    """
    pattern = re.compile(r'<(?P<sign>[+-]?)(?P<label>[A-Z_]+)>')
    files = sorted(glob.glob(str(Path(dirpath) / "*.txt")))
    per_file_matches = {}
    for fp in files:
        try:
            txt = open(fp, 'r', encoding='utf-8').read()
        except Exception as e:
            print(f"Could not read {fp}: {e}")
            continue
        matches = []
        for m in pattern.finditer(txt):
            sign = m.group('sign') or ''
            label = m.group('label')
            matches.append({'pos': m.start(), 'sign': sign, 'label': label, 'file': fp})
        per_file_matches[fp] = matches
    return per_file_matches


def aggregate_counts(per_file_matches, labels):
    counts = {label: {'predicted':0,'fp':0,'fn':0} for label in labels}
    for matches in per_file_matches.values():
        for m in matches:
            label = m['label']
            sign = m['sign']
            if label not in counts:
                counts[label] = {'predicted':0,'fp':0,'fn':0}
            if sign == '-':
                counts[label]['fp'] += 1
            elif sign == '+':
                counts[label]['fn'] += 1
            else:
                counts[label]['predicted'] += 1
    return counts


# Taxonomy labels to aggregate
LABELS = ["INST","INTER","ERR_DESC","ERR_SIM","RECON","PLAUS","CURATE","CORR"]

# Directory with manual-reviewed annotated traces

for DIRPATH in [
    "manual_inspection/tagging_evaluation/joint/eedi_data/deepseek-naive-deepseek-reasoner",
    "manual_inspection/tagging_evaluation/joint/eedi_data/deepseek-naive-cot-deepseek-chat",
    "manual_inspection/tagging_evaluation/joint/eedi_data/openrouter-naive-z-ai_glm-4.7-reasoner",
    "manual_inspection/tagging_evaluation/joint/eedi_data/openrouter-naive-cot-z-ai_glm-4.7-chat"
]:
    print("-"*30)
    print(DIRPATH)
    print("-"*30)

    per_file = load_manual_annotations(DIRPATH)
    counts = aggregate_counts(per_file, LABELS)

    # Compute metrics per label and overall
    rows = []
    total_tp = total_fp = total_fn = 0
    for label in LABELS:
        pred = counts.get(label, {}).get('predicted', 0)
        fp = counts.get(label, {}).get('fp', 0)
        fn = counts.get(label, {}).get('fn', 0)
        tp = max(pred - fp, 0)
        precision = tp / (tp + fp) if (tp + fp) > 0 else None
        recall = tp / (tp + fn) if (tp + fn) > 0 else None
        rows.append({'label':label, 'predicted':pred, 'fp':fp, 'fn':fn, 'tp':tp, 'precision':precision, 'recall':recall})
        total_tp += tp
        total_fp += fp
        total_fn += fn

    overall_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else None
    overall_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else None

    # Display results
    df_metrics = pd.DataFrame(rows)
    print(df_metrics.to_string(index=False))
    print()
    print(f"Overall: TP={total_tp} FP={total_fp} FN={total_fn}")
    if overall_precision is not None:
        print(f"Overall precision: {overall_precision:.3f}")
    else:
        print("Overall precision: N/A")
    if overall_recall is not None:
        print(f"Overall recall: {overall_recall:.3f}")
    else:
        print("Overall recall: N/A")

    # Save metrics for inspection
    out_path = Path(DIRPATH) / "aggregate_label_metrics.csv"
    df_metrics.to_csv(out_path, index=False)
    print(f"Saved per-label metrics to {out_path}")

### Full Annotation

In [ ]:
def annotate_reasoning(args):
    client, reasoning, model_config = args

    try:
        annotator = model_config.get("model")
        answer = annotate_rt(client, model_config, reasoning)
    except Exception as e:
        print(e)
        return ("", "")

    return (annotator, answer)

# Function to process a DataFrame and generate annotations in parallel
def generate_annotations_and_save(client, model_config, run_name, df, category, n_samples=30, ):
    """Generates annotations for `n_samples` traces from `df` and saves them."""
    samples = df.sample(n=min(n_samples, len(df)), random_state=42)
    results = []

    args_list = [(client, trace, model_config) for trace in samples["trace"]]
    with ThreadPoolExecutor(max_workers=8) as executor:
        res_ann = list(executor.map(annotate_reasoning, args_list))

    for idx, row in enumerate(samples.itertuples(index=False)):
        row_dict = row._asdict() if hasattr(row, '_asdict') else dict(zip(samples.columns, row))
        row_dict["annotation"] = res_ann[idx][1]
        row_dict["annotater"] = res_ann[idx][0]
        results.append(row_dict)
    out_df = pd.DataFrame(results)
    out_path = f"eedi_data/joint_results/annotated/{run_name}_{category}_annot.csv"
    out_df.to_csv(out_path, index=False)
    print(f"Saved {len(out_df)} annotations to {out_path}")

def annotate_full(run_name: str, n_samples: int = 30, is_reasoning: bool = True):
    with open(f"eedi_data/joint_results/{run_name}_responses_by_datapointid.json", "r") as f:
        responses = json.load(f)

    results_df = pd.read_csv(f"eedi_data/joint_results/{run_name}_results.csv")

    datapoints = []

    for k,response in responses.items():
        dp = eedi_dataset[int(k)]
        try:
            result = results_df[results_df["Id"] == int(k)].iloc[0]
        except:
            print(f"Could not find {k} in {set(results_df['Id'])}")
        
        trace = response.get("raw_reasoning", "")
        if not is_reasoning:
            trace = response.get("step_by_step")
        datapoints.append((k, dp["Problem"]["Question"], trace, result["proportional_match"], result["number_correct"], result["repetitions"], result["distractors"], dp["Problem"]["Solvable"], result["num_cor_sol_steps_by_datapointid"]))

    df = pd.DataFrame(datapoints, columns=["Id", "problem", "trace", "proportional_match", "number_correct", "repetitions", "distractors", "solvable", "nr_steps_of_solution"])

    unsolvable_df = df[~df["solvable"]]

    high_match_solvable_df = df[(df["proportional_match"] > 0.5) & (df["solvable"])]
    low_match_solvable_df = df[(df["proportional_match"] < 0.5)  & (df["solvable"])]

    high_match_long_problems_solvable_df = df[(df["proportional_match"] > 0.5) & (df["nr_steps_of_solution"] > 4) & (df["solvable"])]
    low_match_long_problems_solvable_df = df[(df["proportional_match"] < 0.5) & (df["nr_steps_of_solution"] > 4) & (df["solvable"])]

    model_config = gpt_4_1
    client = OpenAI(base_url=model_config["base_url"], api_key=os.environ.get(model_config["api_key_var"], None))

    generate_annotations_and_save(client, model_config, run_name, high_match_solvable_df, "high_match_solvable", n_samples=n_samples)
    generate_annotations_and_save(client, model_config, run_name, low_match_solvable_df, "low_match_solvable", n_samples=n_samples)
    generate_annotations_and_save(client, model_config, run_name, high_match_long_problems_solvable_df, "high_match_long_problems_solvable", n_samples=n_samples)
    generate_annotations_and_save(client, model_config, run_name, low_match_long_problems_solvable_df, "low_match_long_problems_solvable", n_samples=n_samples)

In [ ]:
annotate_full("deepseek-naive-deepseek-reasoner", n_samples=30)
annotate_full("deepseek-naive-cot-deepseek-chat", n_samples=30, is_reasoning=False)
annotate_full("openrouter-naive-z-ai_glm-4.7-reasoner", n_samples=30)
annotate_full("openrouter-naive-cot-z-ai_glm-4.7-chat", n_samples=30, is_reasoning=False)

### Parse Annotated Sequences
Parse the annotated sequences into lists of labels 

In [ ]:
# Find all relevant CSVs (adjust path if needed)
tag_labels = [
    "<INTER>",
    "<CORR>",
    "<ERR_DESC>",
    "<INST>",
    "<ERR_SIM>",
    "<PLAUS>",
    "<CURATE>",
    "<RECON>"
]

csv_paths = glob.glob("eedi_data/joint_results/annotated/*_annot.csv")

for path in csv_paths:
    df = pd.read_csv(path)
    sequence_lists = []
    for annotation in df["annotation"].fillna(""):
        matches = []
        # (i) Find all tag label matches with their start index
        for tag in tag_labels:
            for m in re.finditer(re.escape(tag), annotation):
                matches.append((m.start(), tag.strip("<>").lower()))
        # (iii) Sort by start index
        matches.sort(key=lambda x: x[0])
        sequence_lists.append(matches)
    # Add the sequence as a new column (as string for CSV compatibility)
    df["annotation_sequence"] = [str(seq) for seq in sequence_lists]
    out_path = path.replace("_annot.csv", "_annot_parsed.csv")
    df.to_csv(out_path, index=False)
    print(f"Exported parsed file: {out_path}")